# 🧬 Invivo Partners — Biotech Research Agent

**Investment-grade biotech research reports, generated in seconds.**

This notebook runs the full pipeline:
- Enter any disease, drug name, or mechanism
- Get a 12-section investor report with 14 charts
- Covers 13 curated diseases fully; any other disease with a live API key

---
### ⚡ Quick start
1. Run **Cell 1** (setup) — once per session
2. Run **Cell 2** (API key) — paste your key, or skip for curated diseases
3. Run **Cell 3** (generate) — change the disease name and run

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# Run this once at the start of every session.

import subprocess, sys, os

# Install dependencies quietly
print('Installing dependencies...')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
    stdout=subprocess.DEVNULL
)
print('✓ Dependencies ready')

# Add repo root to Python path so the agent package is importable
repo_root = os.path.dirname(os.path.abspath('.'))
if repo_root not in sys.path:
    sys.path.insert(0, '.')
print('✓ Environment configured')

In [ ]:
# ── Cell 2: API Key (optional but recommended) ────────────────────────────────
#
# Without a key: works fully for these 13 diseases:
#   Alzheimer's · KRAS inhibitors · GLP-1/obesity · Rheumatoid arthritis
#   CAR-T · NASH · Sickle cell · Multiple sclerosis · Atopic dermatitis
#   Glioblastoma · SMA · Pancreatic cancer · ALS
#
# With a key: works for ANY disease using live PubMed + ClinicalTrials.gov data
#
# Get a free API key at: https://console.anthropic.com

import os

# Option A — paste your key directly (fine for local use, don't commit this):
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-your-key-here'

# Option B — Codespaces/GitHub Actions secret (already set if configured):
if os.environ.get('ANTHROPIC_API_KEY'):
    print('✓ API key found — full mode enabled (any disease, live data)')
else:
    print('ℹ No API key — curated mode (13 built-in diseases work perfectly)')
    print('  To enable all diseases: set ANTHROPIC_API_KEY above and re-run')

In [ ]:
# ── Cell 3: Generate a report ─────────────────────────────────────────────────
#
# Change DISEASE to anything you want to research.
# Examples:
#   'Alzheimer disease'        'ALS Lou Gehrig disease'
#   'KRAS G12C inhibitors'     'pancreatic cancer'
#   'atopic dermatitis'        'sickle cell disease'
#   'tofersen SOD1'            'GLP-1 obesity'
#   (with API key) 'Huntington disease'  'ovarian cancer'  'PCSK9 inhibitors'

DISEASE = 'ALS Lou Gehrig disease'   # ← change this

# ─────────────────────────────────────────────────────────────────────────────
import sys, os, time
sys.path.insert(0, '.')

from biotech_agent.pipeline import run_sync
from IPython.display import display, HTML, FileLink

print(f'Generating report for: {DISEASE}')
print('─' * 50)
start = time.time()

result = run_sync(
    DISEASE,
    progress_callback=lambda msg: print(f'  {msg}')
)

elapsed = time.time() - start
html_content = result['html']
matched = result['retrieval_stats'].get('matched_disease', 'unknown')

print('─' * 50)
print(f'✓ Done in {elapsed:.1f}s  |  {len(html_content)//1024}KB  |  matched: {matched}')

# Save to file
safe_name = DISEASE.lower().replace(' ', '_').replace('/', '_')[:40]
output_path = f'outputs/{safe_name}_report.html'
os.makedirs('outputs', exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f'\n📄 Report saved: {output_path}')
display(FileLink(output_path, result_html_prefix='⬇️  Download report: '))

In [ ]:
# ── Cell 4: Preview report inline ─────────────────────────────────────────────
# Renders the report inside the notebook (scroll to explore).
# Tip: right-click → Open in New Tab for full-screen view.

from IPython.display import IFrame
IFrame(src=output_path, width='100%', height='900px')

In [ ]:
# ── Cell 5: Batch — generate multiple reports at once ─────────────────────────
# Useful for comparing a disease landscape across indications.

import sys, os, time
sys.path.insert(0, '.')

from biotech_agent.pipeline import run_sync
from IPython.display import display, FileLink

DISEASES = [
    'Alzheimer disease',
    'ALS Lou Gehrig disease',
    'Parkinson disease',       # needs API key
]

os.makedirs('outputs', exist_ok=True)
print(f'Generating {len(DISEASES)} reports...\n')

for disease in DISEASES:
    t = time.time()
    try:
        result = run_sync(disease)
        safe = disease.lower().replace(' ', '_')[:35]
        path = f'outputs/{safe}_report.html'
        with open(path, 'w', encoding='utf-8') as f:
            f.write(result['html'])
        matched = result['retrieval_stats'].get('matched_disease', '?')
        print(f'✓ {disease:<35} {len(result["html"])//1024}KB  [{matched}]  {time.time()-t:.1f}s')
        display(FileLink(path, result_html_prefix='   ⬇️  '))
    except Exception as e:
        print(f'✗ {disease:<35} ERROR: {e}')

print('\n✓ All done. Find reports in the outputs/ folder.')

In [ ]:
# ── Cell 6: Supported curated diseases (no API key needed) ────────────────────

import sys
sys.path.insert(0, '.')
from biotech_agent.retrievers.mock_data import DISEASES, _get_trial_counts

counts = _get_trial_counts()
print('Diseases with full curated data (no API key needed):')
print(f'{"Disease":<30} {"Trials":>7}  {"Notes"}')
print('─' * 70)

disease_notes = {
    'alzheimer':          'lecanemab, donanemab, buntanetap, trontinemab, EVOKE',
    'kras':               'sotorasib, adagrasib, RMC-6236, MRTX1133',
    'glp-1':              'tirzepatide, orforglipron, CagriSema, retatrutide',
    'rheumatoid_arthritis':'upadacitinib, deucravacitinib, nipocalimab',
    'cart':               'Carvykti, CARTITUDE-5, allogeneic CAR-T',
    'nash':               'resmetirom (approved), efruxifermin, pegozafermin',
    'sickle_cell':        'Casgevy (CRISPR), mitapivat, inclacumab',
    'multiple_sclerosis': 'tolebrutinib, fenebrutinib, ublituximab',
    'atopic_dermatitis':  'dupilumab, amlitelimab, povorcitinib, orismilast',
    'glioblastoma':       'vorasidenib, DCVax-L, TTFields combos',
    'sma':                'nusinersen, risdiplam, apitegromab, branaplam',
    'pancreatic_cancer':  'FOLFIRINOX, olaparib, RMC-6236, mRNA-5671',
    'als':                'tofersen, AMX0035 (withdrawn), WVE-004, pridopidine',
}

for disease, count in sorted(counts.items()):
    note = disease_notes.get(disease, '')
    print(f'  {disease:<28} {count:>5}   {note}')

print(f'\nTotal: {sum(counts.values())} real clinical trials across {len(counts)} diseases')
print('\nFor any other disease, add ANTHROPIC_API_KEY and run — uses live PubMed + ClinicalTrials.gov')